# Application of Radial Equilibrium Equation
## Adaptation of the code RE-DES by Lewis (Turbomachines Performance Analysis)
This notebook is for the design of a blade by solving the Radial Equilibrium equation

The radial equilibrium equation can be reordened and written as

$$\boxed{
\frac{\text{d}}{\text{d}r} c_x(r)^2 = 2\left( \omega - \frac{c_\theta(r)}{r}\right)
\frac{\text{d} (rc_\theta(r))}{\text{d}r}
}  \tag{1}
$$

that gives the solution
$$
c_x(r) = \sqrt{f(r) + k} \tag{2}
$$
where
$$
f(r) = \int_{r_h}^r \left( \omega - \frac{c_\theta(r)}{r}\right)\text{d} (rc_\theta(r)) \tag{3}
$$

The aim is, given all the data: $Q$, $\omega$, $r_h$, $r_t$ and the function
$c_\theta(r)$, compute $c_x(r)$ and the angle $\beta_2(r)$ that fullfils the
required flowrate.

We start by initialising the libraries we will need to perform the calculations.

In [19]:
import numpy as np
from scipy.interpolate import CubicSpline
from scipy.integrate import cumulative_trapezoid, trapezoid

The problem data is entered, in our case a fan, and the swirl model is chosen with cthetadata. It can be forced vortex
($c_\theta \propto r$), constant, free vortex ($c_\theta \propto 1/r$) or arbitrary vortex (linear combination of forced and
free vortex)

In [20]:
Dh = 0.3  #m Hub diameter
rh = Dh/2  #m Hub radius
Dt = 1.25 #m Tip diameter
rt = Dt/2  #m Tip radius
rrms = np.sqrt(0.5*(rh*rh+rt*rt))
rdata = np.linspace(rh,rt,10) #vector with the info about the blade length
#Qdata = 1.41e5 #m^3/h
#Qdata = Qdata/3600 # m³/s
Qdata = 39 # m^3/s caudal
omega = 1485  #rpm angular velocity
omega = omega*np.pi/30  #rad/s angular velocity
A = 100*1
B = 7.77
#cthetadata = A*rdata                 ## forced vortex (solid body swirl)
#cthetadata = A*np.ones(rdata.size)   ## constant ctheta
cthetadata = B/rdata                 ## free-vortex
#cthetadata = A*(rdata-rrms) + B/rdata  ## Arbitrary vortex
#cthetadata = 2*omega/3*rdata-9.08/np.sqrt(rdata) ## Another arbitrary vortex

The hub to tip ratio and the rms radius, $r_\text{rms} = \sqrt{\frac{r_h^2+r_t^2}{2}}$ are  computed. Also, $c_{\theta,\text{rms}}=c_\theta(r=r_\text{rms})$ is calculated as an interpolation of given data, and  $c_x$ in $r_\text{rms}$ is as well estimated, assuming that it is the average velocity given by the flow rate, $\overline{c_x}$.
For numerical computations, the $r$ domain is divided in $m$ intervals.

In [21]:
h = rh/rt
ctrms = CubicSpline(rdata,cthetadata)(rrms)
cxm = Qdata/(np.pi*(rt*rt-rh*rh))
m = 601   # number of interpolation points
r = np.linspace(rh,rt,m)
print("Hub to tip ratio = {:0.4f}".format(h))
print("rms = {:.4f} m".format(rrms))
print("ctheta_rms = {:.4f} m/s".format(ctrms))
print("cx_rms = {:.4f} m/s".format(cxm))

Hub to tip ratio = 0.2400
rms = 0.4545 m
ctheta_rms = 17.0958 m/s
cx_rms = 33.7225 m/s


And now, the function $f(r)$ is computed by numerical integration (Eq. (3))

In [22]:
ctheta = CubicSpline(rdata,cthetadata)(r)
f = cumulative_trapezoid(omega-np.divide(ctheta,r),
                         np.multiply(r,ctheta),initial=0)

### First approximation

The first aproximation of the value of $k$ is with the assumption that
$c_{x,\text{rms}} = \overline{c_x}$

In [23]:
frms = CubicSpline(r,f)(rrms)
k = cxm*cxm-frms
print("First approximation of k: \n k = {:.4f} m²/s²".format(k))

First approximation of k: 
 k = 1138.9823 m²/s²


The obtained solution for k is printed.

In the first instance we define $D_F$ as 0.5, a conservative figure, considering that 0.6 would be a perfect performance for the profile. Also, $C_D≈0$ is estimated since the lift is expected to be maximum and the drag is expected to be minimum, so 0.01 is taken as $C_D$.

With the above assumptions and data, ${C_L}$, ${σ}$ (solidity) and ${c_x}$ along the entire length of the profile are calculated.

With these data, the chord of the profile along the length of the blade is calculated, thus defining the change of dimensions that the blade will have.

After obtaining all these data, the blade design is completely closed and only the CAD design is necessary.

$$
σ = \frac {cos(β_1) (tan(β_1) - tan(β_2))}{2 D_F - 2  [1 - \frac{cos(β_1)}{cos(β_2) } ] } \tag{4}
$$

$$
C_L = \frac {2  cos(β_m)  (tan(β_1) - tan(β_2))}{σ} - C_Dtan(β_m) \tag{5}
$$

In [30]:
DF = 0.5
CD = 0.01
def printSolution(k):
    cx = np.sqrt(k + f)
    Q = 2*np.pi*trapezoid(np.multiply(r,cx),r)
    print("{:^15}{:^10}{:^10}{:^10}{:^10}{:^10}{:^10}{:^10}{:^10}{:^10}{:^10}"
          .format("Radius","ctheta","cx","alpha2","beta1","beta2","beta_m","Solidity","CD","CL","chord"))
    print("\u2500"*115)
    for i in range(rdata.size):
        cxans = CubicSpline(r,cx)(rdata[i])
        alpha2 = np.arctan(cthetadata[i]/cxans)
        beta2 = np.arctan((omega*rdata[i]-cthetadata[i])/cxans)
        beta1 = np.arctan(omega*rdata[i]/cxans)
        beta_m = np.arctan(0.5*(np.tan(beta1)+np.tan(beta2)))
        cosbeta1 = np.cos(beta1)
        cosbeta2 = np.cos(beta2)
        tanbeta1 = np.tan(beta1)
        tanbeta2 = np.tan(beta2)
        cosbetam = np.cos(beta_m)
        tanbetam = np.tan(beta_m)
        solidity = (cosbeta1* (tanbeta1 - tanbeta2)) / (2*DF - 2 * (1 - (cosbeta1/cosbeta2) ))
        Nblades = 9
        bladespace = 2*np.pi*rdata[i]/Nblades
        chord =  solidity * bladespace
        CL = (2/solidity) * (cosbetam * (tanbeta1 - tanbeta2)) - CD*tanbetam
        print("{:^15.4f}{:^10.2f}{:^10.2f}{:^10.2f}{:^10.2f}{:^10.2f}{:^10.2f}{:^10.2f}{:^10.2f}{:^10.2f}{:^10.3f}".
              format(rdata[i],
                     cthetadata[i],
                     cxans,
                     np.rad2deg(alpha2),
                     np.rad2deg(beta1),
                     np.rad2deg(beta2),
                     np.rad2deg(beta_m),
                     solidity,
                     CD,
                     CL,
                     chord))
    print("Q = {:.3f} m^3/s".format(Q))
    errorQ = np.abs(Qdata-Q)/Qdata
    print("error = {:.1e} %".format(errorQ))

printSolution(k)

    Radius       ctheta      cx      alpha2    beta1     beta2     beta_m   Solidity     CD        CL      chord   
───────────────────────────────────────────────────────────────────────────────────────────────────────────────────
    0.1500       51.80     33.75     56.91     34.65     -40.15    -4.36      1.10      0.01      2.79     0.115   
    0.2028       38.32     33.72     48.65     43.08     -11.38    20.15      1.69      0.01      1.26     0.240   
    0.2556       30.40     33.72     42.04     49.68     15.48     36.04      1.70      0.01      0.85     0.304   
    0.3083       25.20     33.72     36.77     54.88     34.00     46.35      1.11      0.01      0.92     0.239   
    0.3611       21.52     33.72     32.54     59.01     45.77     53.39      0.69      0.01      1.09     0.174   
    0.4139       18.77     33.72     29.10     62.35     53.51     58.47      0.46      0.01      1.25     0.133   
    0.4667       16.65     33.72     26.28     65.08     58.91     62.30

### More precise computation


To calculate these parameters more accurately, we introduce a new library and recalculate K as a function created instead of the trapezoidally calculated point-to-point area.

In [31]:
from scipy.optimize import brentq

In [32]:
def QFunction(k):
    cx = np.sqrt(k + f)
    Q_temptative = 2*np.pi*trapezoid(np.multiply(r,cx),r)
    return Qdata-Q_temptative

# brentq(f,a,b) finds a root of the function f in the interval (a,b)
k = brentq(QFunction,0.5*k,1.5*k)
print("k = {:.4f} m²/s²".format(k))

k = 1139.0210 m²/s²


A more accurate K is obtained, but as can be seen the values in the table above hardly change because the error previously made was minimal.

In [33]:
printSolution(k)

    Radius       ctheta      cx      alpha2    beta1     beta2     beta_m   Solidity     CD        CL      chord   
───────────────────────────────────────────────────────────────────────────────────────────────────────────────────
    0.1500       51.80     33.75     56.91     34.65     -40.15    -4.36      1.10      0.01      2.79     0.115   
    0.2028       38.32     33.72     48.65     43.08     -11.38    20.15      1.69      0.01      1.26     0.240   
    0.2556       30.40     33.72     42.04     49.68     15.48     36.04      1.70      0.01      0.85     0.304   
    0.3083       25.20     33.72     36.77     54.88     34.00     46.35      1.11      0.01      0.92     0.239   
    0.3611       21.52     33.72     32.54     59.01     45.77     53.39      0.69      0.01      1.09     0.174   
    0.4139       18.77     33.72     29.10     62.35     53.51     58.47      0.46      0.01      1.25     0.133   
    0.4667       16.65     33.72     26.28     65.08     58.91     62.30